In [6]:
# Kaggle environment - adjust dataset name to match your uploaded dataset
import os

# List available input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

DATA_DIR = '/kaggle/input/competitions/playground-series-s6e5'

/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


In [7]:
import pandas as pd

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")

Train: (439140, 16)
Test: (188165, 15)


## Improved Model v3: Optimized Weights + CatBoost Ensemble

In [8]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Reload data (use f-string paths for Kaggle)
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

def add_features(df):
    df = df.copy()
    df['TyreLife_x_Compound'] = df['TyreLife'].astype(str) + '_' + df['Compound']
    df['LapNumber_sq'] = df['LapNumber'] ** 2
    df['TyreLife_sq'] = df['TyreLife'] ** 2
    df['TyreLife_per_Progress'] = df['TyreLife'] / (df['RaceProgress'] + 1e-6)
    df['LapTime_vs_RaceMean'] = df.groupby('Race')['LapTime (s)'].transform(lambda x: x - x.mean())
    df['Stint_x_TyreLife'] = df['Stint'] * df['TyreLife']
    df['HighTyreLife'] = (df['TyreLife'] > 20).astype(int)
    df['PosGroup'] = pd.cut(df['Position'], bins=[0, 5, 10, 20], labels=['top', 'mid', 'back'])
    df['CumPitStops'] = df.groupby(['Race', 'Year', 'Driver'])['PitStop'].cumsum()
    df['LapsToGo'] = df.groupby(['Race', 'Year', 'Driver'])['LapNumber'].transform('max') - df['LapNumber']
    df['Delta_positive'] = (df['LapTime_Delta'] > 0).astype(int)
    df['Abs_LapTime_Delta'] = df['LapTime_Delta'].abs()
    df['RaceLapCount'] = df.groupby(['Race', 'Year'])['LapNumber'].transform('max')
    df['LapInRace'] = df['LapNumber'] / df['RaceLapCount']
    return df

train['is_train'] = 1
test['is_train'] = 0
test['PitNextLap'] = np.nan
combined = pd.concat([train, test], ignore_index=True)
combined = add_features(combined)
train_fe = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test_fe = combined[combined['is_train'] == 0].drop(columns=['is_train', 'PitNextLap'])

# Prepare features
target = "PitNextLap"
drop_cols = ["id", target]
features = [c for c in train_fe.columns if c not in drop_cols]
cat_features = ["Driver", "Compound", "Race", "TyreLife_x_Compound", "PosGroup"]

X = train_fe[features].copy()
y = train_fe[target].copy()
X_test = test_fe[features].copy()

for col in cat_features:
    if col in X.columns:
        X[col] = X[col].astype("category")
        X_test[col] = X_test[col].astype("category")

print(f"Features ({len(features)})")
print(f"Categorical: {cat_features}")
print(f"Train: {X.shape}, Test: {X_test.shape}")

Features (28)
Categorical: ['Driver', 'Compound', 'Race', 'TyreLife_x_Compound', 'PosGroup']
Train: (439140, 28), Test: (188165, 28)


In [9]:
import catboost as cb
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# LightGBM: CPU only (GPU doesn't support high-cardinality categoricals)
lgb_params = dict(
    n_estimators=2000, learning_rate=0.03, max_depth=7, num_leaves=63,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1,
)

xgb_params = dict(
    n_estimators=2000, learning_rate=0.03, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, eval_metric='auc',
    device='cuda',  # GPU
    enable_categorical=True,
)

catboost_params = dict(
    iterations=2000, learning_rate=0.03, depth=7,
    l2_leaf_reg=3.0,
    subsample=0.8, bootstrap_type='Bernoulli',  # Bernoulli required for subsample
    random_seed=42, eval_metric='AUC',
    verbose=0, early_stopping_rounds=100,
    task_type='GPU',  # GPU
)

cat_idx = [features.index(c) for c in cat_features if c in features]

X_xgb = X.copy()
X_test_xgb = X_test.copy()
for col in cat_features:
    if col in X_xgb.columns:
        X_xgb[col] = X_xgb[col].cat.codes
        X_test_xgb[col] = X_test_xgb[col].cat.codes

X_cb = X.copy()
X_test_cb = X_test.copy()
for col in cat_features:
    if col in X_cb.columns:
        X_cb[col] = X_cb[col].astype(str)
        X_test_cb[col] = X_test_cb[col].astype(str)

print("Params and data prepared.")

Params and data prepared.


In [10]:
# 5-fold training loop
val_scores_lgb, val_scores_xgb, val_scores_cb = [], [], []
test_preds_lgb, test_preds_xgb, test_preds_cb = [], [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    # --- LightGBM ---
    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)])
    p_lgb = m_lgb.predict_proba(X_va)[:, 1]
    val_scores_lgb.append(roc_auc_score(y_va, p_lgb))
    test_preds_lgb.append(m_lgb.predict_proba(X_test)[:, 1])

    # --- XGBoost ---
    X_tr_x, X_va_x = X_xgb.iloc[train_idx], X_xgb.iloc[val_idx]
    m_xgb = xgb.XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr_x, y_tr, eval_set=[(X_va_x, y_va)], verbose=False)
    p_xgb = m_xgb.predict_proba(X_va_x)[:, 1]
    val_scores_xgb.append(roc_auc_score(y_va, p_xgb))
    test_preds_xgb.append(m_xgb.predict_proba(X_test_xgb)[:, 1])

    # --- CatBoost ---
    X_tr_c, X_va_c = X_cb.iloc[train_idx], X_cb.iloc[val_idx]
    m_cb = cb.CatBoostClassifier(**catboost_params)
    m_cb.fit(X_tr_c, y_tr, eval_set=[(X_va_c, y_va)], cat_features=cat_idx, verbose=0)
    p_cb = m_cb.predict_proba(X_va_c)[:, 1]
    val_scores_cb.append(roc_auc_score(y_va, p_cb))
    test_preds_cb.append(m_cb.predict_proba(X_test_cb)[:, 1])

    print(f'Fold {fold+1}: LGB={val_scores_lgb[-1]:.5f}, XGB={val_scores_xgb[-1]:.5f}, CB={val_scores_cb[-1]:.5f}')

Training until validation scores don't improve for 100 rounds
[500]	valid_0's binary_logloss: 0.236158
Early stopping, best iteration is:
[840]	valid_0's binary_logloss: 0.235334


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1: LGB=0.94425, XGB=0.95065, CB=0.94970
Training until validation scores don't improve for 100 rounds
[500]	valid_0's binary_logloss: 0.238112
Early stopping, best iteration is:
[655]	valid_0's binary_logloss: 0.237804


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 2: LGB=0.94298, XGB=0.94863, CB=0.94778
Training until validation scores don't improve for 100 rounds
[500]	valid_0's binary_logloss: 0.236609
Early stopping, best iteration is:
[566]	valid_0's binary_logloss: 0.236352


...

Fold 3: LGB=0.94367, XGB=0.94953, CB=0.94864
Training until validation scores don't improve for 100 rounds
[500]	valid_0's binary_logloss: 0.238403
Early stopping, best iteration is:
[747]	valid_0's binary_logloss: 0.237909


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 4: LGB=0.94297, XGB=0.94917, CB=0.94776
Training until validation scores don't improve for 100 rounds
[500]	valid_0's binary_logloss: 0.235074
Early stopping, best iteration is:
[822]	valid_0's binary_logloss: 0.234515


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 5: LGB=0.94463, XGB=0.95007, CB=0.94880


In [11]:
# Ensemble with optimized weights
mean_lgb = np.mean(val_scores_lgb)
mean_xgb = np.mean(val_scores_xgb)
mean_cb = np.mean(val_scores_cb)

total = mean_lgb + mean_xgb + mean_cb
w_lgb = mean_lgb / total
w_xgb = mean_xgb / total
w_cb = mean_cb / total

test_ensemble = (w_lgb * np.mean(test_preds_lgb, axis=0) +
                 w_xgb * np.mean(test_preds_xgb, axis=0) +
                 w_cb * np.mean(test_preds_cb, axis=0))

print(f'LGB Mean: {mean_lgb:.5f} (weight={w_lgb:.3f})')
print(f'XGB Mean: {mean_xgb:.5f} (weight={w_xgb:.3f})')
print(f'CB  Mean: {mean_cb:.5f} (weight={w_cb:.3f})')
print(f'Ensemble weighted AUC: {w_lgb*mean_lgb + w_xgb*mean_xgb + w_cb*mean_cb:.5f}')

LGB Mean: 0.94370 (weight=0.332)
XGB Mean: 0.94961 (weight=0.334)
CB  Mean: 0.94854 (weight=0.334)
Ensemble weighted AUC: 0.94729


In [12]:
# Generate submission
submission = pd.DataFrame({"id": test["id"], "PitNextLap": test_ensemble})
submission.to_csv("submission.csv", index=False)
print(f"Submission saved: {submission.shape}")
submission.head()

Submission saved: (188165, 2)


,id,PitNextLap
0,439140,0.004412
1,439141,0.004858
2,439142,0.003923
3,439143,0.199551
4,439144,0.854284
